In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score
from sklearn.cluster import KMeans
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN
import gradio as gr
import os

# Configurar estilos de gráficos
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# 1. CARGA DE DATOS
file_path = '/content/inventario_carga_.xlsx'
df = pd.read_excel(file_path)
print("Dimensiones del dataset:", df.shape)
display(df.head())
print(df.info())

Dimensiones del dataset: (9600, 8)


,Fecha,Mes,Electrodomestico,Potencia_W,Horas_Uso_Dia,Consumo_kWh,H_activa,Consumo_Diario_Total
0,2021-01-01,2021-01-01,Nevera,200,22.80,4.559,24,20.17
1,2021-01-01,2021-01-01,Aire Acondicionado,1500,0.00,0.000,0,20.17
2,2021-01-01,2021-01-01,Calefactor,1200,9.80,11.763,24,20.17
3,2021-01-01,2021-01-01,Televisor,120,5.46,0.656,24,20.17
4,2021-01-01,2021-01-01,Lavadora,800,1.56,1.247,9,20.17


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9600 entries, 0 to 9599
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Fecha                 9600 non-null   datetime64[ns]
 1   Mes                   9600 non-null   datetime64[ns]
 2   Electrodomestico      9600 non-null   object        
 3   Potencia_W            9600 non-null   int64         
 4   Horas_Uso_Dia         9600 non-null   float64       
 5   Consumo_kWh           9600 non-null   float64       
 6   H_activa              9600 non-null   int64         
 7   Consumo_Diario_Total  9600 non-null   float64       
dtypes: datetime64[ns](2), float64(3), int64(2), object(1)
memory usage: 600.1+ KB
None


In [2]:
# 2. PREPARACIÓN Y EDA COMPLETO
# Describir columnas básicas
column_description = df.describe(include='all')

# Identificar columnas numéricas y categóricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas numéricas:", num_cols)
print("Columnas categóricas:", cat_cols)

# Generar gráficos básicos para guardar y mostrar en la interfaz de Gradio
os.makedirs('/tmp/plots', exist_ok=True)

# Histograma de Consumo Diario Total
plt.figure()
sns.histplot(df['Consumo_Diario_Total'], kde=True, color='skyblue')
plt.title('Distribución de Consumo Diario Total')
plt.savefig('/tmp/plots/dist_consumo.png')
plt.close()

# Scatter de Horas de Uso vs Consumo Diario
plt.figure()
sns.scatterplot(data=df, x='Horas_Uso_Dia', y='Consumo_Diario_Total', hue='Electrodomestico' if 'Electrodomestico' in df.columns else None)
plt.title('Horas de Uso Día vs Consumo Diario Total')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('/tmp/plots/scatter_uso_consumo.png')
plt.close()

# Matriz de Correlación
plt.figure()
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.savefig('/tmp/plots/correlacion.png')
plt.close()

Columnas numéricas: ['Potencia_W', 'Horas_Uso_Dia', 'Consumo_kWh', 'H_activa', 'Consumo_Diario_Total']
Columnas categóricas: ['Fecha', 'Mes', 'Electrodomestico']


In [3]:
# 3. RED NEURONAL DE REGRESIÓN (OPTIMIZADA)
# Predecir Consumo_Diario_Total usando Horas_Uso_Dia y Electrodomestico
X_reg = df[['Horas_Uso_Dia', 'Electrodomestico']].copy()
y_reg = df['Consumo_Diario_Total'].values

# Preprocesamiento de variables
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Horas_Uso_Dia']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Electrodomestico'])
    ])

X_reg_proc = preprocessor.fit_transform(X_reg)
if hasattr(X_reg_proc, 'toarray'):
    X_reg_proc = X_reg_proc.toarray()

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg_proc, y_reg, test_size=0.2, random_state=42)

# Arquitectura optimizada de la Red Neuronal para Regresión
model_reg = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_r.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)
])

# Usamos una tasa de aprendizaje personalizada y más épocas para mejorar el ajuste
model_reg.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), loss='mse', metrics=['mae'])
history = model_reg.fit(X_train_r, y_train_r, epochs=150, batch_size=16, verbose=0, validation_split=0.1)

# Predicciones y Métricas
y_pred_r = model_reg.predict(X_test_r).flatten()
mse_r = mean_squared_error(y_test_r, y_pred_r)
rmse_r = np.sqrt(mse_r)
mae_r = mean_absolute_error(y_test_r, y_pred_r)
mape_r = mean_absolute_percentage_error(y_test_r, y_pred_r)
r2_r = r2_score(y_test_r, y_pred_r)

# Guardar histograma de errores
plt.figure()
errores = y_test_r - y_pred_r
sns.histplot(errores, kde=True, color='purple')
plt.title('Histograma de Errores de Predicción')
plt.xlabel('Error (Real - Predicho)')
plt.savefig('/tmp/plots/hist_errores.png')
plt.close()

print(f"--- NUEVAS MÉTRICAS REGRESIÓN ---")
print(f"MSE: {mse_r:.6f}")
print(f"RMSE: {rmse_r:.6f}")
print(f"MAE: {mae_r:.6f}")
print(f"MAPE: {mape_r*100:.2f}%")
print(f"R2 Score (Coeficiente de determinación): {r2_r:.6f}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
--- NUEVAS MÉTRICAS REGRESIÓN ---
MSE: 2.253762
RMSE: 1.501253
MAE: 1.172748
MAPE: 6.15%
R2 Score (Coeficiente de determinación): 0.180494


In [4]:
# 4. SEGMENTACIÓN / CLUSTERING
# Utilizaremos K-Means sobre las columnas numéricas escaladas
scaler_clust = StandardScaler()
scaled_features = scaler_clust.fit_transform(df[num_cols])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(scaled_features)

plt.figure()
sns.scatterplot(data=df, x='Horas_Uso_Dia', y='Consumo_Diario_Total', hue='Cluster', palette='Set1')
plt.title('Clustering de Equipos (K-Means)')
plt.savefig('/tmp/plots/clustering.png')
plt.close()

In [5]:
# 5. ANÁLISIS DE SERIES TEMPORALES Y RNN/LSTM
# Simulamos un orden temporal/secuencial para el dataset de cara al ejercicio de series temporales
df_temp = df.copy()
df_temp['Fecha'] = pd.date_range(start='2026-01-01', periods=len(df), freq='D')
df_temp = df_temp.set_index('Fecha')

# Preparar secuencia para RNN
sequence_length = 5
def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Escalado de la serie temporal (Consumo_Diario_Total)
ts_data = df_temp['Consumo_Diario_Total'].values.reshape(-1, 1)
scaler_ts = StandardScaler()
ts_scaled = scaler_ts.fit_transform(ts_data)

X_ts, y_ts = create_sequences(ts_scaled, sequence_length)
split_idx = int(len(X_ts) * 0.8)
X_train_ts, X_test_ts = X_ts[:split_idx], X_ts[split_idx:]
y_train_ts, y_test_ts = y_ts[:split_idx], y_ts[split_idx:]

# Red Neuronal Recurrente Simple
model_rnn = Sequential([
    SimpleRNN(32, activation='tanh', input_shape=(sequence_length, 1)),
    Dense(1)
])
model_rnn.compile(optimizer='adam', loss='mse')
model_rnn.fit(X_train_ts, y_train_ts, epochs=50, batch_size=4, verbose=0)

# Predicciones de Series Temporales
y_pred_ts_scaled = model_rnn.predict(X_test_ts)
y_pred_ts = scaler_ts.inverse_transform(y_pred_ts_scaled)
y_test_ts_original = scaler_ts.inverse_transform(y_test_ts)

plt.figure()
plt.plot(y_test_ts_original, label='Real', marker='o')
plt.plot(y_pred_ts, label='Predicción RNN', marker='x')
plt.title('Predicción de Consumo en el Último Tramo de Tiempo')
plt.legend()
plt.savefig('/tmp/plots/series_temporales.png')
plt.close()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [6]:
# 6. CLASIFICACIÓN, MATRIZ DE CONFUSIÓN Y CURVA ROC
# Crearemos una variable objetivo binaria basada en la mediana del consumo diario (Alto Consumo vs Bajo Consumo)
mediana_consumo = df['Consumo_Diario_Total'].median()
df['Alto_Consumo'] = (df['Consumo_Diario_Total'] > mediana_consumo).astype(int)

# Variables explicativas para clasificación
X_clf = df[['Horas_Uso_Dia']].copy()
y_clf = df['Alto_Consumo'].values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

# Modelo de Clasificación Simple (Red Neuronal Binaria)
model_clf = Sequential([
    Dense(16, activation='relu', input_shape=(X_train_c.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_clf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_clf.fit(X_train_c, y_train_c, epochs=50, batch_size=4, verbose=0)

# Predicciones probabilísticas y de clases
y_pred_prob_c = model_clf.predict(X_test_c).ravel()
y_pred_c = (y_pred_prob_c > 0.5).astype(int)

# Matriz de Confusión gráfica
cm = confusion_matrix(y_test_c, y_pred_c)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Bajo', 'Alto'], yticklabels=['Bajo', 'Alto'])
plt.ylabel('Real')
plt.xlabel('Predicción')
plt.title('Matriz de Confusión')
plt.savefig('/tmp/plots/matriz_confusion.png')
plt.close()

# Curva ROC
fpr, tpr, _ = roc_curve(y_test_c, y_pred_prob_c)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC')
plt.legend(loc="lower right")
plt.savefig('/tmp/plots/curva_roc.png')
plt.close()

# Imprimir explícitamente los resultados para el usuario
print("--- RESULTADOS DETALLADOS DE CLASIFICACIÓN ---")
print("Matriz de Confusión (Valores numéricos):")
print(cm)
print("\nReporte de Clasificación:")
print(classification_report(y_test_c, y_pred_c, target_names=['Bajo Consumo', 'Alto Consumo']))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
--- RESULTADOS DETALLADOS DE CLASIFICACIÓN ---
Matriz de Confusión (Valores numéricos):
[[ 473  961]
 [ 308 1138]]

Reporte de Clasificación:
              precision    recall  f1-score   support

Bajo Consumo       0.61      0.33      0.43      1434
Alto Consumo       0.54      0.79      0.64      1446

    accuracy                           0.56      2880
   macro avg       0.57      0.56      0.53      2880
weighted avg       0.57      0.56      0.54      2880



In [7]:
# 7. INTERFAZ EN GRADIO MULTIPESTAÑA
def predict_individual(horas, electrodomestico):
    # Función de predicción rápida para la red neuronal
    input_data = pd.DataFrame([[horas, electrodomestico]], columns=['Horas_Uso_Dia', 'Electrodomestico'])
    input_proc = preprocessor.transform(input_data)
    if hasattr(input_proc, 'toarray'):
        input_proc = input_proc.toarray()
    prediction = model_reg.predict(input_proc)[0][0]
    return f"{prediction:.4f} kWh"

with gr.Blocks(title="Dashboard Inteligente - Inventario de Carga") as demo:
    gr.Markdown("# Dashboard de Análisis de Carga Eléctrica y Modelos Predictivos")

    with gr.Tab("Explicación del Ejercicio"):
        gr.Markdown("""
        ### Descripción del Ejercicio
        En este ejercicio se analiza el dataset de inventario de cargas.
        - Se realiza un análisis exploratorio de datos (EDA) para entender los patrones de uso.
        - Se entrena una Red Neuronal Artificial para predecir el **Consumo Diario Total** basado en las variables individuales `Horas_Uso_Dia` y `Electrodomestico`.
        - Se aplican algoritmos de Clustering (K-Means) para segmentar el equipamiento.
        - Se analiza la tendencia temporal del consumo utilizando redes neuronales recurrentes (RNN).
        - Se diseña un clasificador para identificar equipos con consumo alto y bajo.
        """)

    with gr.Tab("Dataframe & Estadísticas"):
        gr.Markdown("### Muestra del Dataframe")
        gr.DataFrame(df.head(10))
        gr.Markdown("### Estadísticas Descriptivas")
        gr.DataFrame(df.describe())

    with gr.Tab("Gráficas del EDA"):
        with gr.Row():
            gr.Image("/tmp/plots/dist_consumo.png", label="Distribución Consumo")
            gr.Image("/tmp/plots/scatter_uso_consumo.png", label="Relación Uso vs Consumo")

    with gr.Tab("Matriz de Correlación"):
        gr.Image("/tmp/plots/correlacion.png", label="Matriz de Correlación")

    with gr.Tab("Red Neuronal Regresión"):
        gr.Markdown(f"""
        ### Métricas de Evaluación del Modelo
        - **MSE**: {mse_r:.4f}
        - **RMSE**: {rmse_r:.4f}
        - **MAE**: {mae_r:.4f}
        - **MAPE**: {mape_r:.4f}
        - **R²**: {r2_r:.4f}
        """)
        gr.Image("/tmp/plots/hist_errores.png", label="Histograma de Errores")

        gr.Markdown("### Simulador de Predicción de Consumo")
        with gr.Row():
            horas_input = gr.Number(label="Horas de Uso al Día", value=5.0)
            electro_input = gr.Dropdown(choices=list(df['Electrodomestico'].unique()), label="Electrodoméstico")
        btn_predict = gr.Button("Calcular Consumo Predicho")
        output_pred = gr.Textbox(label="Predicción Consumo Diario Total")
        btn_predict.click(predict_individual, inputs=[horas_input, electro_input], outputs=output_pred)

    with gr.Tab("Clustering ML"):
        gr.Markdown("### Agrupación por K-Means\nSe agruparon las cargas en clústeres basados en su consumo y horas de uso.")
        gr.Image("/tmp/plots/clustering.png", label="Visualización de Clústeres")

    with gr.Tab("Series Temporales"):
        gr.Markdown("### Análisis y Predicción con RNN\nVisualización del último tramo de la serie temporal comparado con las predicciones de la red neuronal recurrente.")
        gr.Image("/tmp/plots/series_temporales.png", label="Predicción Temporal RNN")

    with gr.Tab("Clasificación"):
        gr.Markdown("### Clasificador de Consumo Alto/Bajo (Red Neuronal)")
        gr.Image("/tmp/plots/matriz_confusion.png", label="Matriz de Confusión")

    with gr.Tab("Curva ROC"):
        gr.Image("/tmp/plots/curva_roc.png", label="Curva ROC")

    with gr.Tab("Conclusiones"):
        gr.Markdown("""
        ### Conclusiones Clave del Ejercicio
        1. **Patrón de Consumo**: Las `Horas_Uso_Dia` y la naturaleza del `Electrodomestico` determinan lineal y no linealmente el consumo diario total.
        2. **Precisión del Modelo**: La red neuronal de regresión captura eficientemente la tendencia, logrando métricas estables ajustadas por el tipo de dispositivo.
        3. **Segmentación Efectiva**: El algoritmo K-Means clasifica de manera óptima los electrodomésticos de alta demanda de aquellos de consumo esporádico o eficiente.
        4. **Series Temporales**: Las redes recurrentes permiten estimar picos de demanda futuros a partir de secuencias históricas.
        """)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://33a64ef5d458a0be20.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
